In [1]:
!pip install numpy==1.26.4

In [2]:
!pip install surprise

In [3]:
!wget 'https://drive.google.com/uc?id=1m0rwReR09achL0xTM6QPoN4tykz5bOMx' -O MovieLens.zip

--2025-12-21 20:32:24--  https://drive.google.com/uc?id=1m0rwReR09achL0xTM6QPoN4tykz5bOMx
Resolving drive.google.com (drive.google.com)... 192.178.142.102, 192.178.142.101, 192.178.142.139, ...
Connecting to drive.google.com (drive.google.com)|192.178.142.102|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1m0rwReR09achL0xTM6QPoN4tykz5bOMx [following]
--2025-12-21 20:32:24--  https://drive.usercontent.google.com/download?id=1m0rwReR09achL0xTM6QPoN4tykz5bOMx
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.121.132, 2607:f8b0:4023:80b::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.121.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 847695 (828K) [application/octet-stream]
Saving to: ‘MovieLens.zip’

MovieLens.zip       100%[===================>] 827.83K  --.-KB/s    in 0.1s    

2025-12-21 20:32:26 (6.23 M

In [4]:
!unzip MovieLens.zip

Archive:  MovieLens.zip
replace links.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: links.csv               
replace movies.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: movies.csv              
replace ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ratings.csv             
replace tags.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: tags.csv                


In [5]:
import pandas as pd
import numpy as np

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')
ratings.head(), ratings.shape

(   userId  movieId  rating  timestamp
 0       1        1     4.0  964982703
 1       1        3     4.0  964981247
 2       1        6     4.0  964982224
 3       1       47     5.0  964983815
 4       1       50     5.0  964982931,
 (100836, 4))

In [6]:
n_total = len(ratings)
n_total

100836

In [7]:
ratings.columns

Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='object')

In [8]:
needed_cols = ['userId', 'movieId', 'rating']
missing = [c for c in needed_cols if c not in ratings.columns]
missing

[]

In [9]:
n_users = ratings['userId'].nunique()
n_items = ratings['movieId'].nunique()
n_ratings = len(ratings)
density = n_ratings / (n_users * n_items)

(n_users, n_items, n_ratings, float(density))

(610, 9724, 100836, 0.016999683055613623)

In [10]:
from surprise import Dataset, Reader

rating_min = float(ratings['rating'].min())
rating_max = float(ratings['rating'].max())

reader = Reader(rating_scale=(rating_min, rating_max))

data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

(rating_min, rating_max)

(0.5, 5.0)

In [11]:
from surprise.model_selection import cross_validate
from surprise import BaselineOnly

algo_baseline = BaselineOnly()

cv_baseline = cross_validate(
    algo_baseline,
    data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

float(np.mean(cv_baseline['test_rmse'])), float(np.std(cv_baseline['test_rmse']))

Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Estimating biases using als...
Evaluating RMSE of algorithm BaselineOnly on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8683  0.8773  0.8740  0.8614  0.8801  0.8722  0.0067  
Fit time          0.37    0.43    0.44    0.39    0.35    0.39    0.04    
Test time         0.10    0.24    0.10    0.06    0.17    0.14    0.06    


(0.8722124755739646, 0.006687687105839178)

In [12]:
from surprise import SVD

algo_svd = SVD(
    n_factors=100,
    n_epochs=25,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

cv_svd = cross_validate(
    algo_svd,
    data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

rmse_svd_mean = float(np.mean(cv_svd['test_rmse']))
rmse_svd_std = float(np.std(cv_svd['test_rmse']))

rmse_svd_mean, rmse_svd_std

Evaluating RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8660  0.8765  0.8707  0.8799  0.8747  0.8736  0.0048  
Fit time          1.66    1.70    1.71    1.74    2.18    1.80    0.19    
Test time         0.11    0.22    0.12    0.22    0.18    0.17    0.05    


(0.873560893184889, 0.004784930795724471)

In [13]:
from surprise.model_selection import GridSearchCV

param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 30],
    'lr_all': [0.002, 0.005, 0.007],
    'reg_all': [0.02, 0.05]
}

gs = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse'],
    cv=5,
    n_jobs=1
)

gs.fit(data)

gs.best_score['rmse'], gs.best_params['rmse']

(0.8599530364703858,
 {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.007, 'reg_all': 0.05})

In [14]:
best_params = gs.best_params['rmse']

algo_svd_best = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42
)

cv_svd_best = cross_validate(
    algo_svd_best,
    data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

rmse_svd_best_mean = float(np.mean(cv_svd_best['test_rmse']))
rmse_svd_best_std = float(np.std(cv_svd_best['test_rmse']))

rmse_svd_best_mean, rmse_svd_best_std

Evaluating RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8643  0.8589  0.8567  0.8597  0.8531  0.8585  0.0037  
Fit time          2.50    2.55    3.02    2.72    2.51    2.66    0.20    
Test time         0.11    0.27    0.17    0.24    0.12    0.18    0.06    


(0.8585499509711655, 0.0036582192915105695)

In [15]:
from surprise import SVDpp

algo_svdpp = SVDpp(
    n_factors=50,
    n_epochs=20,
    lr_all=0.007,
    reg_all=0.02,
    random_state=42
)

cv_svdpp = cross_validate(
    algo_svdpp,
    data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

rmse_svdpp_mean = float(np.mean(cv_svdpp['test_rmse']))
rmse_svdpp_std = float(np.std(cv_svdpp['test_rmse']))

rmse_svdpp_mean, rmse_svdpp_std

Evaluating RMSE of algorithm SVDpp on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8642  0.8570  0.8755  0.8653  0.8579  0.8640  0.0066  
Fit time          179.09  206.32  208.30  207.33  204.45  201.10  11.08   
Test time         12.62   12.76   12.77   12.58   12.84   12.72   0.10    


(0.8639912633943221, 0.006605966842171333)

In [16]:
rows = []

rows.append({
    'model': 'BaselineOnly',
    'rmse_mean': float(np.mean(cv_baseline['test_rmse'])),
    'rmse_std': float(np.std(cv_baseline['test_rmse']))
})

rows.append({
    'model': 'SVD (manual params)',
    'rmse_mean': rmse_svd_mean,
    'rmse_std': rmse_svd_std
})

rows.append({
    'model': 'SVD (best gridsearch)',
    'rmse_mean': rmse_svd_best_mean,
    'rmse_std': rmse_svd_best_std
})

if 'cv_svdpp' in globals():
    rows.append({
        'model': 'SVD++',
        'rmse_mean': rmse_svdpp_mean,
        'rmse_std': rmse_svdpp_std
    })

df_rmse = pd.DataFrame(rows).sort_values('rmse_mean').reset_index(drop=True)

df_rmse

,model,rmse_mean,rmse_std
0,SVD (best gridsearch),0.858550,0.003658
1,SVD++,0.863991,0.006606
2,BaselineOnly,0.872212,0.006688
3,SVD (manual params),0.873561,0.004785


In [17]:
best_rmse = float(df_rmse.iloc[0]['rmse_mean'])
best_model = df_rmse.iloc[0]['model']

best_model, best_rmse, best_rmse <= 0.87

('SVD (best gridsearch)', 0.8585499509711655, True)

## Итоговый вывод

В данной работе была построена и оценена модель рекомендательной системы на датасете MovieLens с использованием библиотеки `surprise`. Для оценки качества применялась метрика RMSE, рассчитанная корректно через 5-fold cross-validation, что даёт устойчивую оценку качества и снижает зависимость результата от случайного разбиения на train/test.

В качестве базового ориентира была обучена модель `BaselineOnly`, после чего были протестированы модели матричной факторизации `SVD` и `SVD++`. Для `SVD` выполнен подбор гиперпараметров с помощью `GridSearchCV`. По результатам сравнения лучшей оказалась модель **SVD с параметрами, подобранными grid search**, показавшая средний **RMSE = 0.8586** (std ≈ 0.0037) на 5 фолдах. Это значение удовлетворяет требованию задания **RMSE ≤ 0.87**.